In [1]:
from cellpose import models, core, io, utils
from bioio import BioImage
import napari
from skimage.transform import resize, rescale
from skimage.measure import regionprops_table

In [2]:
test_img = r"E:\KristinG\20260625_zebrafish\fin\DKO_ctrl_1_skin_1_ventral_fin_Out.czi"
img = BioImage(test_img)
img_rd = img.get_image_data("ZYX", C=1)

In [ ]:
img_twoch = img.get_image_data("ZYXC")

In [ ]:
img_twoch.shape

In [12]:
cp = io.logger_setup()
model = models.CellposeModel(gpu=True)

[GUI INFO] : WRITING LOG OUTPUT TO C:\Users\gkristin\.cellpose\run.log

cellpose version: 	4.2.1.1 
platform:       	win32 
python version: 	3.12.13 
torch version:  	2.12.1+cu126
2026-06-26 17:46:05,644 [io INFO] WRITING LOG OUTPUT TO C:\Users\gkristin\.cellpose\run.log
2026-06-26 17:46:05,645 [io INFO] 
cellpose version: 	4.2.1.1 
platform:       	win32 
python version: 	3.12.13 
torch version:  	2.12.1+cu126
2026-06-26 17:46:05,846 [core INFO] ** TORCH CUDA version installed and working. **
2026-06-26 17:46:05,849 [core INFO] >>>> using GPU (CUDA)
2026-06-26 17:46:07,946 [models INFO] >>>> loading model C:\Users\gkristin\.cellpose\models\cpsam_v2


In [ ]:
v = napari.Viewer()

In [ ]:
v.add_image(img_rd)
v.add_labels(masks, name='masks', opacity=0.5)

In [14]:
rescaled_img = rescale(img_rd, [0.5,0.2,0.2], anti_aliasing=True, channel_axis=None,preserve_range=True)

In [ ]:
rescaled_img_twoch = rescale(img_twoch, 0.15, anti_aliasing=True, channel_axis=3,preserve_range=True)

In [15]:
rescaled_img.shape

(22, 289, 289)

In [ ]:
v.add_image(rescaled_img_twoch, name='rescaled image twoch', scale=(img.physical_pixel_sizes[0]*(200/30), img.physical_pixel_sizes[1]*(200/30), img.physical_pixel_sizes[2]*(200/30)),channel_axis=3)

In [16]:
new_scale = img.physical_pixel_sizes[0]*0.5, img.physical_pixel_sizes[1]*0.2, img.physical_pixel_sizes[2]*0.2

In [17]:
anisotropy = new_scale[0]/new_scale[1]

In [ ]:
rescaled_masks, rescaled_flows, _ = model.eval(rescaled_img, z_axis=0, batch_size=32, do_3D=True, flow3D_smooth=3,cellprob_threshold=-2.0,normalize=True,anisotropy=anisotropy)

In [23]:
rescaled_masks_stitch, rescaled_flows_stitch, _ = model.eval(rescaled_img, z_axis=0, batch_size=32, do_3D=False,normalize=True,stitch_threshold=0.7,anisotropy=anisotropy)

2026-06-26 17:53:55,687 [utils INFO] 100%|##########| 3/3 [00:03<00:00,  1.22s/it]
2026-06-26 17:53:55,691 [models INFO] network run in 3.66s
2026-06-26 17:53:57,853 [utils INFO] 100%|##########| 22/22 [00:02<00:00, 10.19it/s]
2026-06-26 17:53:57,856 [models INFO] stitching 22 planes using stitch_threshold=0.700 to make 3D masks


100%|██████████| 21/21 [00:00<00:00, 482.58it/s]

2026-06-26 17:53:57,952 [models INFO] masks created in 2.26s


In [19]:
v = napari.Viewer()

In [24]:
#v.add_image(rescaled_img, name='rescaled image', scale=(img.physical_pixel_sizes[0]*0.5, img.physical_pixel_sizes[1]*0.2, img.physical_pixel_sizes[2]*0.2))
v.add_labels(rescaled_masks_stitch, name='rescaled masks stitch 0.7 norm anisotropy', opacity=0.5,scale=(img.physical_pixel_sizes[0]*0.5, img.physical_pixel_sizes[1]*0.2, img.physical_pixel_sizes[2]*0.2))

<Labels layer 'rescaled masks stitch 0.7 norm anisotropy' at 0x15acc841520>

In [25]:
props = regionprops_table(rescaled_masks_stitch, properties=['label'])

In [26]:
#v2.add_image(rescaled_img, name='rescaled image', scale=(img.physical_pixel_sizes[0]*(200/30), img.physical_pixel_sizes[1]*(200/30), img.physical_pixel_sizes[2]*(200/30)))
for l in props['label']:
    obj = rescaled_masks_stitch == l
    v.add_labels(obj, name=f'object {l}', scale=(img.physical_pixel_sizes[0]*0.5, img.physical_pixel_sizes[1]*0.2, img.physical_pixel_sizes[2]*0.2))
    if l == 15:
        break

In [ ]:
for l in props['equivalent_diameter_area']:
    print(l)